# SolarGuard AI — Baseline CNN
### CAI 421 · Checkpoint 2 · Reproducible Notebook

**Goal:** Train a 12-class CNN baseline on the Raptor Maps InfraredSolarModules thermal images. Compare future models under the **same data splits**.

**Notebook roadmap:** 1. Load dataset → 2. Audit labels and duplicates → 3. Split → 4. Preprocess → 5. CNN → 6. Train → 7. Test → 8. Save results.

> **Previous run (already completed; NOT generated by this notebook):** Test Accuracy **0.7497**, Test Macro-F1 **0.5605**, best validation Macro-F1 **0.5438**. These results were from the user's earlier run. Running the training cells again will produce a new experiment, possibly with different scores. The seed below applies to **new training runs**.

> **Data note:** Image mode `L`, PIL image size `(width=24, height=40)`, PyTorch tensor shape `(1, 40, 24)`. The model retains the same convolutional structure as the team's architecture; only the dimension order is corrected.


## 1. Imports and reproducibility
Import libraries once. Set the seed **before creating the model or data loaders**.

In [ ]:
import os, json, zipfile, hashlib, random, copy
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

# Fix random seeds for repeatable experiments
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 2. Download and extract the dataset
The ZIP file comes from the official Raptor Maps GitHub repository (MIT License). We do not download it again if it already exists.

In [ ]:
# Clone the dataset repository only once
if not os.path.exists("InfraredSolarModules/2020-02-14_InfraredSolarModules.zip"):
    !git clone https://github.com/RaptorMaps/InfraredSolarModules.git

# Extract only when the metadata is missing
zip_path = "InfraredSolarModules/2020-02-14_InfraredSolarModules.zip"
dataset_path = "solar_dataset/InfraredSolarModules"
if not os.path.exists(os.path.join(dataset_path, "module_metadata.json")):
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("solar_dataset")

# Inspect images and labels
with open(os.path.join(dataset_path, "module_metadata.json"), "r") as f:
    metadata = json.load(f)

df = pd.DataFrame(metadata.values())
images_path = os.path.join(dataset_path, "images")
image_files = [f for f in os.listdir(images_path) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
print("Image files:", len(image_files))
print("Metadata entries:", len(df))
print("Classes:", df["anomaly_class"].nunique())
print(df["anomaly_class"].value_counts().sort_index())


## 3. Quality check: exact duplicates and conflicting labels
**Why?** The same image must not appear in both training and testing, and an identical image with two different labels is ambiguous. This procedure only checks **byte-identical files**; near-duplicates and location-level leakage are not ruled out. All removals are documented, and original files remain unchanged.

In [ ]:
# Create a SHA-256 fingerprint for every image file
def image_hash(relative_path):
    full_path = os.path.join(dataset_path, relative_path)
    with open(full_path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

df_clean = df.copy()
df_clean["image_hash"] = df_clean["image_filepath"].apply(image_hash)

# Find same-content images with different labels
conflict_counts = df_clean.groupby("image_hash")["anomaly_class"].nunique()
conflict_hashes = set(conflict_counts[conflict_counts > 1].index)
conflict_rows = df_clean[df_clean["image_hash"].isin(conflict_hashes)].copy()
conflict_rows.to_csv("conflicting_labels.csv", index=False)

# Exclude ambiguous groups and retain one copy per remaining image
without_conflicts = df_clean[~df_clean["image_hash"].isin(conflict_hashes)].copy()
df_final = without_conflicts.drop_duplicates(subset="image_hash").copy()

print("Conflicting hash groups:", len(conflict_hashes))
print("Conflicting rows excluded:", len(df_clean) - len(without_conflicts))
print("Additional exact duplicates removed:", len(without_conflicts) - len(df_final))
print("Final images:", len(df_final))
print("Final classes:", df_final["anomaly_class"].nunique())
print(df_final["anomaly_class"].value_counts().sort_index())


## 4. Stratified 70/15/15 split
Training teaches the CNN. Validation selects the best epoch. Test is used **only once for final evaluation**. Save the exact split lists so all four models can use the same images.

In [ ]:
# Split only the cleaned dataset; never use the earlier unclean split
train_df, temp_df = train_test_split(
    df_final, test_size=0.30, random_state=SEED, stratify=df_final["anomaly_class"])
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["anomaly_class"])

# Verify exact content does not cross splits
hash_sets = [set(x["image_hash"]) for x in (train_df, val_df, test_df)]
assert len(hash_sets[0] & hash_sets[1]) == 0
assert len(hash_sets[0] & hash_sets[2]) == 0
assert len(hash_sets[1] & hash_sets[2]) == 0

# Save each image path and its class; image_hash is for auditing only
for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split[["image_filepath", "anomaly_class", "image_hash"]].to_csv(f"{name}_split.csv", index=False)

print("Train / Validation / Test:", len(train_df), len(val_df), len(test_df))
print("All splits are free of identical-image overlap")


## 5. Preprocessing and PyTorch DataLoaders
Images are grayscale and normalized to **[0, 1]**. The input tensor is `(channel=1, height=40, width=24)`. No test augmentation is applied in the baseline.

In [ ]:
class_names = sorted(df_final["anomaly_class"].unique())
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

# Load one image as a 1 × 40 × 24 tensor
def load_image(image_path):
    with Image.open(os.path.join(dataset_path, image_path)) as image:
        pixels = np.asarray(image.convert("L"), dtype=np.float32) / 255.0
    return torch.from_numpy(pixels).unsqueeze(0)

class SolarDataset(Dataset):
    def __init__(self, dataframe):
        self.rows = dataframe.reset_index(drop=True)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows.iloc[index]
        image = load_image(row["image_filepath"])
        label = class_to_idx[row["anomaly_class"]]
        return image, label

train_dataset = SolarDataset(train_df)
val_dataset = SolarDataset(val_df)
test_dataset = SolarDataset(test_df)

# Shuffle training samples only
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

images, labels = next(iter(train_loader))
print("Batch shape:", images.shape)
print("Class mapping:", class_to_idx)


## 6. Baseline CNN architecture
Match Alaa’s proposed backbone: Conv 16 → Conv 32 → Conv 64, each with BatchNorm, ReLU, and 2×2 MaxPool. **No attention**, and the training loss will be Cross-Entropy. Spatial output: `64 × 5 × 3 = 960` values per image.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(960, 64), nn.ReLU(),
            nn.Dropout(0.30), nn.Linear(64, 12),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = BaselineCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)

# Validate the architecture dimensions
model.eval()
with torch.no_grad():
    output = model(images.to(device))
print("Input:", images.shape, "Output:", output.shape)
assert output.shape == (images.size(0), 12)


Input: torch.Size([64, 1, 40, 24]) Output: torch.Size([64, 12])


## 7. Training and validation
**Settings:** AdamW, learning rate `0.001`, weight decay `0.0001`, 50 epochs maximum, early stopping patience 8, selected by **validation Macro-F1**.

> **Important:** Running this cell trains a **new model** and can take several minutes. The previously reported 74.97% and 56.05% belong to the earlier experiment; do not copy them as results of a newly seeded run. If using the previously downloaded `SolarGuard_Baseline.pth`, upload it to Colab and load it instead of re-training (see the optional cell below).

In [ ]:
# Set to True to train a NEW model, or False to skip this training cell
RUN_TRAINING = True
best_f1 = -1.0
best_weights = None
wait = 0
history = []

if RUN_TRAINING:
    for epoch in range(50):
        model.train()
        loss_total = 0.0
        for batch_images, batch_labels in train_loader:
            batch_images = batch_images.to(device)
            batch_labels = batch_labels.to(device)
            optimizer.zero_grad()
            scores = model(batch_images)
            loss = criterion(scores, batch_labels)
            loss.backward()
            optimizer.step()
            loss_total += loss.item() * batch_images.size(0)

        # Calculate Macro-F1 on validation data only
        model.eval()
        true_labels, predictions = [], []
        with torch.no_grad():
            for batch_images, batch_labels in val_loader:
                scores = model(batch_images.to(device))
                true_labels.extend(batch_labels.tolist())
                predictions.extend(scores.argmax(dim=1).cpu().tolist())
        val_f1 = f1_score(true_labels, predictions, average="macro", zero_division=0)
        avg_loss = loss_total / len(train_dataset)
        history.append({"epoch": epoch + 1, "train_loss": avg_loss, "val_macro_f1": val_f1})
        print(f"Epoch {epoch+1}: Train Loss={avg_loss:.4f}, Val Macro-F1={val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_weights = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
        if wait >= 8:
            print("Early stopping at epoch", epoch + 1)
            break

    model.load_state_dict(best_weights)
    pd.DataFrame(history).to_csv("training_history.csv", index=False)
    print("Best validation Macro-F1:", round(best_f1, 4))
else:
    print("Training skipped. Load previously trained weights before evaluation.")


### Optional: load the earlier saved model instead of training again
Run this **only** if you have uploaded the previous `SolarGuard_Baseline.pth` into the Colab session. This is useful for recovering the original experiment. Do not run after new training unless you intentionally want to replace the new model with the old one.

In [ ]:
# OPTIONAL — uncomment only when restoring the previously saved model
# from google.colab import files
# files.upload()  # Select SolarGuard_Baseline.pth
# checkpoint = torch.load("SolarGuard_Baseline.pth", map_location=device, weights_only=True)
# model.load_state_dict(checkpoint["model_state_dict"])
# best_f1 = checkpoint["best_val_macro_f1"]
# print("Restored previous model; validation Macro-F1:", best_f1)


## 8. Held-out test evaluation
Report **Accuracy**, **Macro-F1**, and **per-class Recall**, plus a confusion matrix. Do not use the test set to choose hyperparameters.

In [ ]:
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for batch_images, batch_labels in test_loader:
        scores = model(batch_images.to(device))
        y_true.extend(batch_labels.tolist())
        y_pred.extend(scores.argmax(dim=1).cpu().tolist())

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
report = classification_report(
    y_true, y_pred, labels=list(range(12)), target_names=class_names,
    output_dict=True, zero_division=0)

print("Test Accuracy:", round(accuracy, 4))
print("Test Macro-F1:", round(macro_f1, 4))
print(classification_report(
    y_true, y_pred, labels=list(range(12)), target_names=class_names, zero_division=0))

# Save the detailed report
with open("SolarGuard_Baseline_Results.json", "w") as f:
    json.dump(report, f, indent=4)


## 9. Confusion matrix
Rows are **true classes** and columns are **predicted classes**. Diagonal cells show correct predictions.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(12)))
fig, ax = plt.subplots(figsize=(12, 10))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, cmap="Blues", xticks_rotation=90, colorbar=False)
plt.tight_layout()
plt.savefig("SolarGuard_Confusion_Matrix.png", dpi=300, bbox_inches="tight")
plt.show()


## 10. Save the model, metrics, and experiment settings
The `.pth` file stores trained weights and metadata; the `.json` report contains test metrics; the CSV files preserve splits. Download the files before the Colab session ends.

In [ ]:
# Save the trained model and experimental settings
checkpoint = {
    "model_state_dict": model.state_dict(),
    "class_names": class_names,
    "best_val_macro_f1": float(best_f1) if best_f1 >= 0 else None,
    "test_accuracy": float(accuracy),
    "test_macro_f1": float(macro_f1),
    "seed": SEED,
    "training_images": len(train_df),
    "validation_images": len(val_df),
    "test_images": len(test_df),
    "optimizer": "AdamW", "learning_rate": 0.001,
    "weight_decay": 0.0001, "batch_size": 64,
    "loss": "CrossEntropyLoss",
}
torch.save(checkpoint, "SolarGuard_Baseline.pth")

with open("Baseline_Summary.txt", "w") as f:
    f.write("SolarGuard AI — Baseline CNN Results\n")
    f.write(f"Train / Val / Test: {len(train_df)} / {len(val_df)} / {len(test_df)}\n")
    f.write(f"Best validation Macro-F1: {best_f1:.4f}\n")
    f.write(f"Test Accuracy: {accuracy:.4f}\n")
    f.write(f"Test Macro-F1: {macro_f1:.4f}\n")
print("Saved model, report, splits, and summary.")


## 11. Optional downloads (Google Colab)
Run this cell **only when you want to download outputs**. Downloads are not automatic, so `Runtime → Run all` does not create many download dialogs.

In [ ]:
# OPTIONAL: remove the # from one or more lines to download files
# from google.colab import files
# files.download("SolarGuard_Baseline.pth")
# files.download("SolarGuard_Baseline_Results.json")
# files.download("SolarGuard_Confusion_Matrix.png")
# files.download("Baseline_Summary.txt")
# files.download("train_split.csv")
# files.download("val_split.csv")
# files.download("test_split.csv")
# files.download("conflicting_labels.csv")
# files.download("training_history.csv")


## 12. Interpretation and limitations

- **Interpretation:** Compare accuracy with macro-F1 and per-class recall. Strong accuracy can hide weak detection of rare classes.
- **Fair comparisons:** The attention and Focal Loss variants must use the **same splits, preprocessing, backbone, and evaluation protocol**.
- **Remaining checks:** Exact duplicates were checked; near-duplicate and capture-site/session overlap require further review when metadata or better matching methods are available.
- **Scope:** Brightness, contrast, and Gaussian noise tests are planned for later experiments and **are not results of this baseline notebook**.
- **Academic integrity:** Numbers reported in the final paper must match real runs. Review and explain all methods in your own words.

**Dataset source and license:** https://github.com/RaptorMaps/InfraredSolarModules (MIT).
